# Bước 6: Hệ thống gợi ý đa phương tiện
## Đề 13: Phân tích hội thoại trong DVKH trực tuyến
**Mục tiêu:** Xây dựng hệ thống gợi ý câu trả lời + ảnh minh họa cho tư vấn viên

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import cosine_similarity
import re
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('../data/final/classified_data.csv')
rules_df = pd.read_csv('../data/final/association_rules.csv') if pd.io.common.file_exists('../data/final/association_rules.csv') else pd.DataFrame()
slang_dict = pd.read_csv('../data/processed/slang_dict.csv')
slang_map = dict(zip(slang_dict['slang'], slang_dict['standard']))

print(f'Dataset: {len(df)} dòng')
print(f'Luật kết hợp: {len(rules_df)}')

## 6.1 Xây dựng các module

In [ ]:
# ==========================================
# MODULE 1: Tiền xử lý
# ==========================================
def preprocess(text, slang_map):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Zàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    words = [slang_map.get(w, w) for w in words]
    return ' '.join(words)


# ==========================================
# MODULE 2: Train models
# ==========================================
customer_df = df[df['speaker'] == 'customer'].copy()
agent_df = df[df['speaker'] == 'agent'].copy()

# TF-IDF cho toàn bộ customer messages
tfidf = TfidfVectorizer(max_features=3000, min_df=2, max_df=0.95, ngram_range=(1, 2))
X_all = tfidf.fit_transform(customer_df['message_clean'].fillna(''))

# Gộp theo conv_id để train Naive Bayes
conv_text = customer_df.groupby('conv_id').agg(
    text=('message_clean', lambda x: ' '.join(x.dropna().astype(str))),
    satisfaction=('final_satisfaction', 'first'),
    topic=('topic_name', 'first'),
    cluster=('cluster_id', 'first')
).reset_index()
conv_text = conv_text[conv_text['text'].str.strip() != '']

X_conv = tfidf.transform(conv_text['text'])

# Train Naive Bayes
nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_conv, conv_text['satisfaction'])
print(f'✅ Naive Bayes trained on {len(conv_text)} conversations')


# ==========================================
# MODULE 3: Tìm câu trả lời tương tự
# ==========================================
# Xây dựng kho câu trả lời từ agent
agent_responses = []
for conv_id in df['conv_id'].unique():
    conv = df[df['conv_id'] == conv_id]
    customer_msgs = ' '.join(conv[conv['speaker'] == 'customer']['message_clean'].dropna().astype(str))
    agent_msgs = conv[conv['speaker'] == 'agent']['message'].tolist()
    topic = conv['topic_name'].iloc[0]
    cluster = conv['cluster_id'].iloc[0] if 'cluster_id' in conv.columns else -1
    
    if customer_msgs.strip() and agent_msgs:
        agent_responses.append({
            'conv_id': conv_id,
            'customer_text': customer_msgs,
            'agent_response': agent_msgs[0],  # Lấy câu trả lời đầu tiên
            'all_responses': ' | '.join(agent_msgs),
            'topic': topic,
            'cluster': cluster
        })

response_df = pd.DataFrame(agent_responses)
X_responses = tfidf.transform(response_df['customer_text'])
print(f'✅ Response database: {len(response_df)} entries')

## 6.2 Ánh xạ cụm → Ảnh minh họa

In [ ]:
# Mapping chủ đề/cụm → ảnh minh họa
IMAGE_MAP = {
    'giao hàng': 'assets/suggestion_images/giao_hang.png',
    'hoàn tiền': 'assets/suggestion_images/hoan_tien.png',
    'đổi trả': 'assets/suggestion_images/doi_tra.png',
    'kỹ thuật': 'assets/suggestion_images/ky_thuat.png',
    'thanh toán': 'assets/suggestion_images/thanh_toan.png',
    'bảo hành': 'assets/suggestion_images/bao_hanh.png',
}

def get_image(topic):
    topic_lower = str(topic).lower()
    for key, path in IMAGE_MAP.items():
        if key in topic_lower:
            return path
    return 'assets/suggestion_images/giao_hang.png'  # default

print('Image mapping:', IMAGE_MAP)

## 6.3 Hàm gợi ý chính

In [ ]:
def recommend(query, top_k=3):
    """
    Hệ thống gợi ý đa phương tiện
    Input: câu hỏi khách hàng (text thô)
    Output: dict chứa gợi ý câu trả lời + ảnh + phân tích
    """
    # 1. Tiền xử lý
    cleaned = preprocess(query, slang_map)
    
    # 2. Vectorize
    query_vec = tfidf.transform([cleaned])
    
    # 3. Naive Bayes → Mức hài lòng dự đoán
    satisfaction = nb_model.predict(query_vec)[0]
    satisfaction_proba = nb_model.predict_proba(query_vec)[0]
    
    # 4. Tìm câu trả lời tương tự nhất (cosine similarity)
    similarities = cosine_similarity(query_vec, X_responses).flatten()
    top_indices = similarities.argsort()[-top_k:][::-1]
    
    suggestions = []
    for idx in top_indices:
        row = response_df.iloc[idx]
        suggestions.append({
            'response': row['agent_response'],
            'topic': row['topic'],
            'similarity': round(similarities[idx], 4),
            'image': get_image(row['topic'])
        })
    
    # 5. Xác định chủ đề chính
    main_topic = suggestions[0]['topic'] if suggestions else 'Không xác định'
    
    return {
        'query': query,
        'cleaned': cleaned,
        'predicted_satisfaction': satisfaction,
        'satisfaction_proba': dict(zip(nb_model.classes_, satisfaction_proba.round(3))),
        'topic': main_topic,
        'suggestions': suggestions,
        'image': get_image(main_topic)
    }

print('✅ Hàm recommend() sẵn sàng')

## 6.4 Demo hệ thống

In [ ]:
test_queries = [
    'đơn hàng e đặt 3 ngày r mà k thấy ship, sao v ạ',
    'sp bị lỗi muốn hoàn tiền',
    'mua áo size M mà gửi size L, đổi giúm e vs',
    'voucher SALE20 k dùng dc, lỗi j v shop',
    'mình muốn mua laptop cho con học, tầm 15 triệu',
    'app bị crash k thanh toán dc',
    'tài khoản bị khóa k đăng nhập dc',
]

for query in test_queries:
    result = recommend(query)
    print('=' * 70)
    print(f'📩 Khách hàng: {result["query"]}')
    print(f'🔍 Sau xử lý : {result["cleaned"]}')
    print(f'📊 Chủ đề    : {result["topic"]}')
    print(f'😊 Hài lòng  : {result["predicted_satisfaction"]} {result["satisfaction_proba"]}')
    print(f'🖼️  Ảnh minh họa: {result["image"]}')
    print(f'💡 Gợi ý câu trả lời:')
    for i, s in enumerate(result['suggestions'], 1):
        print(f'   [{i}] (similarity={s["similarity"]}) {s["response"][:100]}')
    print()

## 6.5 Đánh giá hệ thống

In [ ]:
# Đánh giá bằng cách lấy mẫu từ dataset, kiểm tra topic prediction
test_sample = conv_text.sample(min(50, len(conv_text)), random_state=42)
correct_topic = 0

for _, row in test_sample.iterrows():
    result = recommend(row['text'])
    # Kiểm tra topic gợi ý có trùng topic gốc không
    if str(result['topic']).lower() in str(row['topic']).lower() or \
       str(row['topic']).lower() in str(result['topic']).lower():
        correct_topic += 1

topic_accuracy = correct_topic / len(test_sample) * 100
print(f'Topic matching accuracy: {topic_accuracy:.1f}% ({correct_topic}/{len(test_sample)})')
print()

# Thời gian phản hồi
import time
times = []
for _, row in test_sample.head(20).iterrows():
    start = time.time()
    _ = recommend(row['text'])
    times.append(time.time() - start)

print(f'Thời gian phản hồi trung bình: {np.mean(times)*1000:.1f}ms')
print(f'Thời gian phản hồi max: {np.max(times)*1000:.1f}ms')

## 6.6 Lưu kết quả & Tổng kết

In [ ]:
# Lưu main_dataset cuối cùng
df.to_csv('../data/final/main_dataset.csv', index=False, encoding='utf-8-sig')
print('✅ Saved: main_dataset.csv')

print()
print('=' * 60)
print('  TỔNG KẾT DỰ ÁN ĐỀ 13')
print('=' * 60)
print(f'Dataset          : {df["conv_id"].nunique()} đoạn, {len(df)} lượt lời')
print(f'Tiền xử lý       : Chuẩn hóa {len(slang_map)} từ lóng + TF-IDF {X_all.shape[1]} features')
print(f'DBSCAN           : {df["cluster_id"].nunique()} cụm')
print(f'Naive Bayes      : Trained on {len(conv_text)} conversations')
print(f'Luật kết hợp     : {len(rules_df)} luật')
print(f'Hệ thống gợi ý  : Top 3 câu trả lời + ảnh minh họa')
print(f'Topic accuracy   : {topic_accuracy:.1f}%')
print(f'Response time    : {np.mean(times)*1000:.1f}ms')
print('=' * 60)